# ket — quantum circuit simulator

This notebook uses the [Deno](https://deno.com/) Jupyter kernel.
Install it with `deno jupyter --install`, then open this file in JupyterLab or VS Code.

All imports resolve from the CDN — no `npm install` needed.

In [ ]:
import { Circuit, qft, iqft, grover, vqe } from 'https://unpkg.com/ket/dist/ket.js'

## Bell state

In [ ]:
const bell = new Circuit(2).h(0).cnot(0, 1)

console.log(bell.draw())
console.log('State:', bell.stateAsString())
console.log('Probs:', bell.exactProbs())

## Shot-based sampling

In [ ]:
const result = bell
  .creg('out', 2)
  .measure(0, 'out', 0)
  .measure(1, 'out', 1)
  .run({ shots: 1000, seed: 42 })

console.log(result.render())

## Grover's search

In [ ]:
// Oracle marks |11⟩ with a phase flip via CZ
// iterations: 1 is exact for N=4 (2-qubit search space)
const search = grover(2, c => c.cz(0, 1), 1)

console.log(search.draw())
console.log('Probabilities after amplification:', search.exactProbs())

// 3-qubit Grover — mark |111⟩ with CCZ = H·CCX·H
const search3 = grover(3, c => c.h(2).ccx(0, 1, 2).h(2))
console.log('3-qubit top result:', search3.run({ shots: 1024, seed: 1 }).most)

## Quantum Fourier Transform

In [ ]:
console.log('QFT(4):')
console.log(qft(4).draw())

// IQFT undoes QFT exactly
const state = new Circuit(3).h(0).x(2)
const rt = new Circuit(3)
  .defineGate('prep', state)
  .defineGate('qft',  qft(3))
  .defineGate('iqft', iqft(3))
  .gate('prep', 0, 1, 2)
  .gate('qft',  0, 1, 2)
  .gate('iqft', 0, 1, 2)

const orig      = state.exactProbs()
const restored  = rt.exactProbs()
const maxDiff   = Math.max(...Object.keys(orig).map(k => Math.abs((restored[k] ?? 0) - orig[k])))
console.log('Max probability difference after QFT→IQFT:', maxDiff.toExponential(2))

## Quantum teleportation

In [ ]:
// Unitary version: no mid-circuit measurement, uses CX/CZ corrections
function teleport(prep) {
  return new Circuit(3)
    .defineGate('payload', prep)
    .gate('payload', 0)
    .h(1).cnot(1, 2)     // Bell pair
    .cnot(0, 1).h(0)     // Alice's Bell measurement
    .cx(1, 2).cz(0, 2)   // Bob's corrections
}

// P(q2 = |0⟩) marginalised over q0/q1
function pQ2is0(probs) {
  return Object.entries(probs).reduce((s, [bs, p]) => bs[0] === '0' ? s + p : s, 0)
}

const cases = [
  { label: '|0⟩',        prep: new Circuit(1),               expected: 1.0 },
  { label: '|1⟩',        prep: new Circuit(1).x(0),          expected: 0.0 },
  { label: '|+⟩',        prep: new Circuit(1).h(0),          expected: 0.5 },
  { label: 'Rx(π/3)|0⟩', prep: new Circuit(1).rx(Math.PI/3, 0), expected: Math.cos(Math.PI/6)**2 },
]

for (const { label, prep, expected } of cases) {
  const p0    = pQ2is0(teleport(prep).exactProbs())
  const ok    = Math.abs(p0 - expected) < 1e-9 ? '✓' : '✗'
  console.log(`${ok}  ${label.padEnd(14)}  P(q2=|0⟩) = ${p0.toFixed(6)}  (expected ${expected.toFixed(6)})`)
}

## Noise models and density matrix

In [ ]:
const pure  = bell.dm()
const noisy = bell.dm({ noise: 'aria-1' })

console.log('Pure Bell state:')
console.log('  purity  =', pure.purity().toFixed(6),  '(1 = pure)')
console.log('  entropy =', pure.entropy().toFixed(6),  'bits')

console.log('\naria-1 noise profile:')
console.log('  purity  =', noisy.purity().toFixed(6))
console.log('  entropy =', noisy.entropy().toFixed(6), 'bits')
console.log('  probs   =', noisy.probabilities())

console.log('\nPurity by device profile:')
for (const p of ['aria-1', 'forte-1', 'harmony']) {
  console.log(`  ${p.padEnd(10)}`, bell.dm({ noise: p }).purity().toFixed(6))
}

## VQE — variational energy minimisation

In [ ]:
// H = 0.5·Z + 0.5·X, ground state energy = -1/√2 ≈ -0.7071
const H = [{ coeff: 0.5, ops: 'Z' }, { coeff: 0.5, ops: 'X' }]

let best = { theta: 0, energy: Infinity }
for (let i = 0; i <= 32; i++) {
  const theta  = (i / 32) * 2 * Math.PI
  const energy = vqe(new Circuit(1).ry(theta, 0), H)
  if (energy < best.energy) best = { theta, energy }
}

console.log('Ground state energy:', best.energy.toFixed(8))
console.log('Exact minimum:      ', (-Math.SQRT2 / 2).toFixed(8))
console.log('Optimal θ:          ', (best.theta / Math.PI).toFixed(4) + 'π')

## Import / Export

In [ ]:
const c = new Circuit(2).h(0).cnot(0, 1)

console.log('── OpenQASM 2.0 ────────────────────────')
console.log(c.toQASM())

console.log('── Qiskit ──────────────────────────────')
console.log(c.toQiskit())

console.log('── Cirq ────────────────────────────────')
console.log(c.toCirq())

console.log('── Quil ────────────────────────────────')
console.log(c.toQuil())

console.log('── LaTeX (quantikz) ────────────────────')
console.log(c.toLatex())

// Lossless round-trip
const json      = c.toJSON()
const restored  = Circuit.fromJSON(json)
const match     = JSON.stringify(c.exactProbs()) === JSON.stringify(restored.exactProbs())
console.log('\nJSON round-trip match:', match)